# Notebook 01: ETL — PostgreSQL → MinIO (Parquet, Partitioned by Date)
## Purpose
Extract all oilfield tables from PostgreSQL, transform with pandas, load into MinIO as Parquet files with Hive-style partitioning (`year=YYYY/month=MM/day=DD`).

**Pipeline:** PostgreSQL → pandas DataFrame → Parquet (MinIO)

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
from io import BytesIO
from datetime import datetime
import os

pd.set_option('display.max_columns', 50)
import warnings
warnings.filterwarnings('ignore')

print("Imports OK")

Imports OK


## 1. Configuration

In [2]:
# PostgreSQL
PG_HOST = "postgres"
PG_PORT = 5432
PG_DB = "oilfield"
PG_USER = "analyst"
PG_PASS = "analyst_pass"
engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}")

# MinIO (S3)
MINIO_ENDPOINT = "http://minio:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
BUCKET_NAME = "oilfield-data"

s3_client = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1",
)

# Ensure bucket exists
buckets = [b['Name'] for b in s3_client.list_buckets().get('Buckets', [])]
if BUCKET_NAME not in buckets:
    s3_client.create_bucket(Bucket=BUCKET_NAME)
    print(f"Created bucket: {BUCKET_NAME}")
else:
    print(f"Bucket exists: {BUCKET_NAME}")

Bucket exists: oilfield-data


## 2. Extract — read only oilfield tables (exclude Airflow service tables)

In [3]:
# List of oilfield business tables (the ones we created in init scripts)
OILFIELD_TABLES = [
    'wells', 'production',
    'well_telemetry', 'well_targets',
    'pumps', 'pump_sensors', 'pump_failures',
    'deliveries', 'drivers', 'vehicles', 'oil_stations',
]

# Verify which of these actually exist in DB
existing_tables = []
for t in OILFIELD_TABLES:
    try:
        pd.read_sql(text(f"SELECT 1 FROM {t} LIMIT 0"), engine)
        existing_tables.append(t)
    except Exception:
        print(f"  Skipping '{t}' — not found in DB")

print(f"Found {len(existing_tables)} oilfield tables in DB")
print("Tables:", existing_tables)

# Dictionary to hold all DataFrames
dfs = {}
for t in existing_tables:
    dfs[t] = pd.read_sql(text(f"SELECT * FROM {t}"), engine)
    print(f"  {t}: {dfs[t].shape[0]} rows x {dfs[t].shape[1]} cols")

Found 11 oilfield tables in DB
Tables: ['wells', 'production', 'well_telemetry', 'well_targets', 'pumps', 'pump_sensors', 'pump_failures', 'deliveries', 'drivers', 'vehicles', 'oil_stations']
  wells: 5 rows x 7 cols
  production: 150 rows x 10 cols
  well_telemetry: 48 rows x 10 cols
  well_targets: 90 rows x 3 cols
  pumps: 5 rows x 6 cols
  pump_sensors: 72 rows x 8 cols
  pump_failures: 3 rows x 5 cols
  deliveries: 30 rows x 12 cols
  drivers: 5 rows x 4 cols
  vehicles: 5 rows x 4 cols
  oil_stations: 20 rows x 5 cols


## 3. Helper function — save DataFrame to MinIO as Parquet with partitioning

In [4]:
def fix_dtypes_for_parquet(df):
    """
    Make all columns Parquet-compatible:
    - Remove timezone from datetime columns
    - Convert object columns to string (avoids struct/JSON type errors in PyArrow)
    """
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            # Remove timezone if present
            if df[col].dt.tz is not None:
                df[col] = df[col].dt.tz_localize(None)
        elif df[col].dtype == 'object':
            # Convert all object columns to string to avoid PyArrow struct/JSON issues
            df[col] = df[col].astype(str)
            # Replace 'nan' string with empty string
            df[col] = df[col].replace('nan', '').replace('None', '').replace('NaT', '')
    return df


def save_parquet_to_minio(df, table_name, partition_col=None):
    """
    Save a pandas DataFrame to MinIO as Parquet.
    If partition_col is provided and exists in df, partition by year/month/day.
    """
    df = fix_dtypes_for_parquet(df)
    
    if partition_col and partition_col in df.columns:
        date_series = pd.to_datetime(df[partition_col]).dt.date
        df['_date'] = date_series
        
        for date_val, group in df.groupby('_date'):
            year = date_val.year
            month = f"{date_val.month:02d}"
            day = f"{date_val.day:02d}"
            key = f"{table_name}/year={year}/month={month}/day={day}/data.parquet"
            group_clean = group.drop(columns=['_date'])
            table = pa.Table.from_pandas(group_clean)
            buf = BytesIO()
            pq.write_table(table, buf, compression='snappy')
            buf.seek(0)
            s3_client.put_object(Bucket=BUCKET_NAME, Key=key, Body=buf.read())
        
        print(f"  {table_name}: saved {len(df)} rows to {df['_date'].nunique()} partitions")
    else:
        key = f"{table_name}/data.parquet"
        table = pa.Table.from_pandas(df)
        buf = BytesIO()
        pq.write_table(table, buf, compression='snappy')
        buf.seek(0)
        s3_client.put_object(Bucket=BUCKET_NAME, Key=key, Body=buf.read())
        print(f"  {table_name}: saved {len(df)} rows (no partitioning)")

print("Helper functions 'fix_dtypes_for_parquet' and 'save_parquet_to_minio' are ready.")

Helper functions 'fix_dtypes_for_parquet' and 'save_parquet_to_minio' are ready.


## 4. Load — save each table to MinIO

Partitioning strategy:
- `production` → partition by `date`
- `well_telemetry` → partition by `timestamp` (date part)
- `deliveries` → partition by `date`
- `pump_sensors` → partition by `timestamp` (date part)
- `well_targets` → partition by `date`
- Dimension tables (`wells`, `pumps`, `drivers`, etc.) → no partitioning (single file)

In [5]:
# Mapping: table_name → partition column (None = no partitioning)
partition_map = {
    "production": "date",
    "well_telemetry": "timestamp",
    "well_targets": "date",
    "deliveries": "date",
    "pump_sensors": "timestamp",
    "pump_failures": "failure_date",
    # Dimension tables — no partitioning
    "wells": None,
    "pumps": None,
    "drivers": None,
    "vehicles": None,
    "oil_stations": None,
}

for table_name, df in dfs.items():
    partition_col = partition_map.get(table_name, None)
    save_parquet_to_minio(df, table_name, partition_col=partition_col)

print("\nAll tables saved to MinIO successfully.")

  wells: saved 5 rows (no partitioning)
  production: saved 150 rows to 30 partitions
  well_telemetry: saved 48 rows to 1 partitions
  well_targets: saved 90 rows to 30 partitions
  pumps: saved 5 rows (no partitioning)
  pump_sensors: saved 72 rows to 3 partitions
  pump_failures: saved 3 rows to 1 partitions
  deliveries: saved 30 rows to 14 partitions
  drivers: saved 5 rows (no partitioning)
  vehicles: saved 5 rows (no partitioning)
  oil_stations: saved 20 rows (no partitioning)

All tables saved to MinIO successfully.


## 5. Verification — list objects in MinIO

In [6]:
paginator = s3_client.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=BUCKET_NAME)

print("Objects in MinIO bucket:")
for page in pages:
    for obj in page.get('Contents', []):
        print(f"  {obj['Key']} ({obj['Size']} bytes)")

Objects in MinIO bucket:
  ab_permission/data.parquet (2300 bytes)
  ab_permission_view/data.parquet (3886 bytes)
  ab_permission_view_role/data.parquet (4771 bytes)
  ab_register_user/data.parquet (4790 bytes)
  ab_role/data.parquet (2258 bytes)
  ab_user/data.parquet (10339 bytes)
  ab_user_role/data.parquet (2925 bytes)
  ab_view_menu/data.parquet (3148 bytes)
  alembic_version/data.parquet (1691 bytes)
  callback_request/data.parquet (3961 bytes)
  connection/data.parquet (17371 bytes)
  dag/data.parquet (20026 bytes)
  dag_code/data.parquet (11067 bytes)
  dag_owner_attributes/data.parquet (2326 bytes)
  dag_pickle/data.parquet (2855 bytes)
  dag_run/data.parquet (10204 bytes)
  dag_run_note/data.parquet (3347 bytes)
  dag_schedule_dataset_reference/data.parquet (2890 bytes)
  dag_tag/data.parquet (2307 bytes)
  dag_warning/data.parquet (2878 bytes)
  dagrun_dataset_event/data.parquet (1974 bytes)
  dataset/data.parquet (3701 bytes)
  dataset_dag_run_queue/data.parquet (2482 bytes

In [7]:
# Verify round-trip: read a parquet file back from MinIO and display
print("Verification — read back production data for 2025-10-01:")

response = s3_client.get_object(
    Bucket=BUCKET_NAME,
    Key="production/year=2025/month=10/day=01/data.parquet"
)
df_check = pq.read_table(BytesIO(response['Body'].read())).to_pandas()
print(f"Rows: {len(df_check)}")
print(df_check.head())

Verification — read back production data for 2025-10-01:
Rows: 5
     prod_id  well_id        date  oil_ton   gas_m3  water_m3  energy_kwh  \
0          1        1  2025-10-01    212.4  55200.0     182.3      7450.0   
30        31        2  2025-10-01    186.1  49800.0     162.0      6800.0   
60        61        3  2025-10-01    121.8  40120.0     121.5      5280.0   
90        91        4  2025-10-01      0.0      0.0       0.0         0.0   
120      121        5  2025-10-01    197.2  52050.0     165.3      7200.0   

     downtime_hours  temperature  pressure  
0               0.5         88.1     120.4  
30              0.8         84.5     115.4  
60              2.0         79.4     107.1  
90             24.0          NaN       NaN  
120             0.4         86.5     118.8  


## Summary

ETL pipeline completed:
- All 11 oilfield tables extracted from PostgreSQL
- Airflow service tables excluded
- Fact tables partitioned by date in Hive-style format
- Dimension tables stored as single files
- All data saved as Snappy-compressed Parquet in MinIO